# RiskBands — exemplo simples para apresentação

Este notebook mostra um fluxo mínimo para começar a usar o **RiskBands** em um cenário de risco de crédito.

A ideia é demonstrar, de forma simples:

1. criar uma base sintética de crédito;
2. ajustar binnings para variáveis explicativas;
3. inspecionar cortes, event rate, WoE e IV;
4. tratar valores ausentes com uma política explícita;
5. transformar os dados;
6. exportar evidências para revisão técnica.

> Este notebook é didático. A base é sintética e não representa dados reais de crédito.

## 0. Kernel e instalação

Antes de executar:

- No VS Code, selecione o kernel do ambiente do projeto, por exemplo `.venv-v210-spark`;
- Em Databricks, use `%pip install riskbands==2.4.0`;
- Localmente, se estiver dentro do repositório, você pode usar `pip install -e .`.

Se precisar instalar a versão publicada:

```python
%pip install riskbands==2.4.0
```

In [22]:
import sys
from pathlib import Path

print("Python:", sys.version)
print("Executável:", sys.executable)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Executável: d:\0_CienciaDados\1_Frameworks\RiskBands\.venv-v210-spark\Scripts\python.exe


## 1. Imports principais

In [23]:
import numpy as np
import pandas as pd

from riskbands import RiskBands, Binner
import riskbands

print("RiskBands version:", riskbands.__version__)
print("RiskBands is Binner:", RiskBands is Binner)

RiskBands version: 2.4.0
RiskBands is Binner: True


## 2. Criar uma base sintética simples

Vamos criar uma base com variáveis comuns em crédito:

- `bureau_score`: score externo ou bureau;
- `income`: renda;
- `channel`: canal de aquisição;
- `target`: evento de inadimplência.

A variável `target` será binária:

- `1`: cliente ruim / evento;
- `0`: cliente bom / não evento.

In [24]:
rng = np.random.default_rng(42)

n = 10_000

bureau_score = rng.normal(loc=620, scale=90, size=n).clip(300, 900)
income = rng.lognormal(mean=8.2, sigma=0.45, size=n).clip(800, 15000)
channel = rng.choice(["branch", "digital", "partner"], size=n, p=[0.35, 0.45, 0.20])

# Introduz missing values de forma controlada
bureau_missing = rng.random(n) < 0.08
income_missing = rng.random(n) < 0.10
channel_missing = rng.random(n) < 0.05

bureau_score = bureau_score.astype("float")
income = income.astype("float")
bureau_score[bureau_missing] = np.nan
income[income_missing] = np.nan

channel = channel.astype("object")
channel[channel_missing] = None

# Risco sintético: score menor e renda menor aumentam a probabilidade de evento
score_component = np.nan_to_num((650 - bureau_score) / 160, nan=0.25)
income_component = np.nan_to_num((3500 - income) / 3500, nan=0.20)
channel_component = np.where(channel == "partner", 0.18, np.where(channel == "digital", 0.03, 0.08))

logit = -1.2 + 1.25 * score_component + 0.85 * income_component + channel_component
prob_default = 1 / (1 + np.exp(-logit))
target = rng.binomial(1, prob_default)

df = pd.DataFrame({
    "bureau_score": bureau_score,
    "income": income,
    "channel": channel,
    "target": target,
})

df.head()

,bureau_score,income,channel,target
0,647.424537,3941.531633,digital,0
1,526.401430,5456.733163,partner,1
2,687.540608,1877.090448,branch,0
3,704.650824,NaN,partner,1
4,444.406833,5287.596020,partner,1


## 3. Leitura rápida da base

Antes do binning, vale olhar:

- tamanho da base;
- taxa geral do target;
- percentual de missing por variável.

In [25]:
print("Shape:", df.shape)
print("Target rate:", round(df["target"].mean(), 4))

missing_summary = (
    df[["bureau_score", "income", "channel"]]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "variable"})
)

missing_summary

Shape: (10000, 4)
Target rate: 0.2998


,variable,missing_rate
0,bureau_score,0.0791
1,income,0.1004
2,channel,0.0536


## 4. Primeiro ajuste: uma variável numérica

Começamos com apenas uma variável para facilitar a leitura.

Neste exemplo, usamos:

```python
missing_policy="merge"
missing_merge_criterion="nearest_event_rate"
```

Isso significa que o grupo missing será fundido ao bin regular com **event rate mais próximo**, quando possível.

In [26]:
rb_score = RiskBands(
    max_bins=5,
    missing_policy="merge",
    missing_merge_criterion="nearest_event_rate",
    missing_merge_fallback="separate_bin",
)

rb_score.fit(
    df,
    y="target",
    column="bureau_score",
)

print("Fit concluído.")

Fit concluído.


## 5. Inspecionar o binning

A tabela de binning mostra:

- bins/cortes;
- quantidade de registros;
- event rate;
- WoE;
- IV ou componentes relacionados, quando disponíveis.

Para apresentação, foque principalmente em:

- os cortes;
- a tendência do event rate;
- a decisão sobre missing.

In [27]:
rb_score.binning_table(column="bureau_score")

,variable,bin,count,event,non_event,event_rate,bin_order,bin_code,share,woe,iv_component
0,bureau_score,"(-inf, 518.58)",1233,712,521,0.577453,0,0.0,0.1233,-1.159844,1.890643e-01
1,bureau_score,"[518.58, 560.39)",1105,448,657,0.405430,1,1.0,0.1105,-0.465238,2.587041e-02
2,bureau_score,"[560.39, 653.85)",4407,1322,3085,0.299977,2,2.0,0.4407,-0.000584,1.502924e-07
3,bureau_score,"[653.85, 736.69)",2355,429,1926,0.182166,3,3.0,0.2355,0.653065,8.613573e-02
4,bureau_score,"[736.69, inf)",900,87,813,0.096667,4,4.0,0.0900,1.381933,1.201975e-01


## 6. Decisão sobre missing values

O `missing_decision_log_` registra a decisão tomada para valores ausentes.

Esta é uma das partes mais importantes para auditabilidade.

In [35]:
decision_log = pd.DataFrame(rb_score.missing_decision_log_)

cols = [
    "variable",
    "action",
    "status",
    "selected_bin_label",
    "missing_merge_criterion",
    "distance_metric",
    "distance",
    "event_rate_missing_fit",
    "selected_bin_event_rate",
    "woe_missing_fit",
    "selected_bin_woe",
    "fallback_used",
    "policy_requested",
]

display(decision_log[[c for c in cols if c in decision_log.columns]].T)

,0
variable,bureau_score
action,missing_merged
status,merged
selected_bin_label,"[560.39, 653.85)"
missing_merge_criterion,nearest_event_rate
distance_metric,abs_event_rate_diff
distance,0.007269
event_rate_missing_fit,0.305942
selected_bin_event_rate,0.298673
woe_missing_fit,-0.029672


In [36]:
rb_score.missing_merge_candidates_

,variable,criterion,candidate_rank,candidate_bin_id,candidate_bin_label,candidate_bin_order,candidate_n,candidate_events,candidate_non_events,candidate_event_rate,...,missing_n,missing_events,missing_non_events,missing_event_rate,missing_woe,distance_event_rate,distance_woe,distance,distance_metric,selected
0,bureau_score,nearest_event_rate,1,2.0,"[560.39, 653.85)",2,3616.0,1080.0,2536.0,0.298673,...,791.0,242.0,549.0,0.305942,-0.029672,0.007269,0.035354,0.007269,abs_event_rate_diff,True
1,bureau_score,nearest_event_rate,2,1.0,"[518.58, 560.39)",1,1105.0,448.0,657.0,0.405430,...,791.0,242.0,549.0,0.305942,-0.029672,0.099488,0.435471,0.099488,abs_event_rate_diff,False
2,bureau_score,nearest_event_rate,3,3.0,"[653.85, 736.69)",3,2355.0,429.0,1926.0,0.182166,...,791.0,242.0,549.0,0.305942,-0.029672,0.123776,0.682831,0.123776,abs_event_rate_diff,False
3,bureau_score,nearest_event_rate,4,4.0,"[736.69, inf)",4,900.0,87.0,813.0,0.096667,...,791.0,242.0,549.0,0.305942,-0.029672,0.209275,1.411700,0.209275,abs_event_rate_diff,False
4,bureau_score,nearest_event_rate,5,0.0,"(-inf, 518.58)",0,1233.0,712.0,521.0,0.577453,...,791.0,242.0,549.0,0.305942,-0.029672,0.271512,1.130078,0.271512,abs_event_rate_diff,False


## 7. Transformar os dados

Depois do fit, podemos transformar a variável original em bins.

Isso é o que geralmente será enviado para modelagem, validação ou análise posterior.

In [9]:
transformed_score = rb_score.transform(
    df[["bureau_score"]],
    column="bureau_score",
    return_type="dataframe",
)

transformed_score.head(10)

,bureau_score
0,"[560.39, 653.85)"
1,"[518.58, 560.39)"
2,"[653.85, 736.69)"
3,"[653.85, 736.69)"
4,"(-inf, 518.58)"
5,"(-inf, 518.58)"
6,"[560.39, 653.85)"
7,"[560.39, 653.85)"
8,"[560.39, 653.85)"
9,"[518.58, 560.39)"


In [37]:
transformed_score["bureau_score"].value_counts(dropna=False, normalize=True)

bureau_score
[560.39, 653.85)    0.4407
[653.85, 736.69)    0.2355
(-inf, 518.58)      0.1233
[518.58, 560.39)    0.1105
[736.69, inf)       0.0900
Name: proportion, dtype: float64

## 8. Ajuste com várias variáveis

Agora usamos três variáveis ao mesmo tempo:

- `bureau_score`;
- `income`;
- `channel`.

Esse é um cenário mais próximo de um fluxo real de seleção e documentação de variáveis.

In [11]:
features = ["bureau_score", "income", "channel"]

rb = RiskBands(
    max_bins=5,
    missing_policy="merge",
    missing_merge_criterion="nearest_event_rate",
    missing_merge_fallback="separate_bin",
)

rb.fit(
    df,
    y="target",
    columns=features,
)

print("Features ajustadas:", features)

Features ajustadas: ['bureau_score', 'income', 'channel']


## 9. Resumo das variáveis

O resumo ajuda a enxergar rapidamente quais variáveis foram ajustadas e suas principais métricas.

In [12]:
rb.summary()

,variable,n_bins,iv,temporal_score,score_strategy,objective_direction,objective_score,objective_preference_score,coverage_ratio_min,coverage_ratio_mean,rare_bin_count,ranking_reversal_period_count,selection_basis,alert_flags
0,bureau_score,5,0.422146,0.0,standard,maximize,0.137751,0.137751,NaN,NaN,0,0,discrimination-led,
1,channel,1,0.000000,0.0,standard,maximize,-0.026000,-0.026000,NaN,NaN,0,0,no_objective_signal,
2,income,5,0.163625,0.0,standard,maximize,0.047269,0.047269,NaN,NaN,0,0,discrimination-led,


## 10. Tabela consolidada de auditoria

A tabela de auditoria consolida informações úteis para revisão técnica.

In [13]:
rb.audit_table()

,dataset,variable,candidate_name,selected_strategy,cut_summary,n_bins,iv,temporal_score,score_strategy,objective_direction,...,objective_raw_event_rate_std_max,objective_raw_woe_std_mean,objective_raw_woe_std_max,objective_raw_bin_share_std_mean,objective_raw_bin_share_std_max,objective_raw_monotonic_break_period_count,objective_raw_ranking_reversal_period_count,objective_raw_bins_missing_any_period_count,objective_raw_missing_period_count,objective_raw_low_coverage_bin_count
0,fit,bureau_score,selected_candidate,supervised,"(-inf, 518.58) | [518.58, 560.39) | [560.39, 6...",5,0.422146,0.0,standard,maximize,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,fit,channel,selected_candidate,supervised,1,1,0.000000,0.0,standard,maximize,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,fit,income,selected_candidate,supervised,"(-inf, 3242.65) | [3242.65, 3654.94) | [3654.9...",5,0.163625,0.0,standard,maximize,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 11. Decisões de missing para todas as variáveis

Aqui fica claro que missing values não são tratados de forma invisível: cada decisão fica registrada.

In [14]:
pd.DataFrame(rb.missing_decision_log_)

,variable,policy_requested,effective_policy,merge_criterion,missing_merge_criterion,missing_merge_fallback,missing_detected,n_missing_fit,event_rate_missing_fit,woe_missing_fit,...,tie_break_applied,tie_detected,candidate_count,candidate_bins,metrics_before,metrics_after,action,status,reason,notes
0,bureau_score,merge,merge,nearest_event_rate,nearest_event_rate,separate_bin,True,791,0.305942,-0.029672,...,False,False,5,"[{'candidate_rank': 1, 'bin_label': '[560.39, ...","{'missing': {'n': 791.0, 'events': 242.0, 'non...","{'merged_bin': {'n': 4407.0, 'events': 1322.0,...",missing_merged,merged,None,Missing values were merged into the nearest ne...
1,income,merge,merge,nearest_event_rate,nearest_event_rate,separate_bin,True,1004,0.373506,-0.331011,...,False,False,5,"[{'candidate_rank': 1, 'bin_label': '(-inf, 32...","{'missing': {'n': 1004.0, 'events': 375.0, 'no...","{'merged_bin': {'n': 4544.0, 'events': 1710.0,...",missing_merged,merged,None,Missing values were merged into the nearest ne...
2,channel,merge,merge,nearest_event_rate,nearest_event_rate,separate_bin,True,536,0.305970,-0.030543,...,False,False,3,"[{'candidate_rank': 1, 'bin_label': 1, 'bin_or...","{'missing': {'n': 536.0, 'events': 164.0, 'non...","{'merged_bin': {'n': 4812.0, 'events': 1434.0,...",missing_merged,merged,None,Missing values were merged into the nearest ne...


## 12. Candidatos avaliados para merge de missing

Quando `missing_policy="merge"` é usado, o RiskBands registra os candidatos considerados para receber o grupo missing.

Isso ajuda a responder:

> Por que o missing foi fundido com este bin e não com outro?

In [15]:
rb.missing_merge_candidates_.head(20)

,variable,criterion,candidate_rank,candidate_bin_id,candidate_bin_label,candidate_bin_order,candidate_n,candidate_events,candidate_non_events,candidate_event_rate,...,missing_n,missing_events,missing_non_events,missing_event_rate,missing_woe,distance_event_rate,distance_woe,distance,distance_metric,selected
0,bureau_score,nearest_event_rate,1,2.0,"[560.39, 653.85)",2,3616.0,1080.0,2536.0,0.298673,...,791.0,242.0,549.0,0.305942,-0.029672,0.007269,0.035354,0.007269,abs_event_rate_diff,True
1,bureau_score,nearest_event_rate,2,1.0,"[518.58, 560.39)",1,1105.0,448.0,657.0,0.405430,...,791.0,242.0,549.0,0.305942,-0.029672,0.099488,0.435471,0.099488,abs_event_rate_diff,False
2,bureau_score,nearest_event_rate,3,3.0,"[653.85, 736.69)",3,2355.0,429.0,1926.0,0.182166,...,791.0,242.0,549.0,0.305942,-0.029672,0.123776,0.682831,0.123776,abs_event_rate_diff,False
3,bureau_score,nearest_event_rate,4,4.0,"[736.69, inf)",4,900.0,87.0,813.0,0.096667,...,791.0,242.0,549.0,0.305942,-0.029672,0.209275,1.411700,0.209275,abs_event_rate_diff,False
4,bureau_score,nearest_event_rate,5,0.0,"(-inf, 518.58)",0,1233.0,712.0,521.0,0.577453,...,791.0,242.0,549.0,0.305942,-0.029672,0.271512,1.130078,0.271512,abs_event_rate_diff,False
5,income,nearest_event_rate,1,0.0,"(-inf, 3242.65)",0,3540.0,1335.0,2205.0,0.377119,...,1004.0,375.0,629.0,0.373506,-0.331011,0.003613,0.015019,0.003613,abs_event_rate_diff,True
6,income,nearest_event_rate,2,1.0,"[3242.65, 3654.94)",1,971.0,298.0,673.0,0.306900,...,1004.0,375.0,629.0,0.373506,-0.331011,0.066606,0.297051,0.066606,abs_event_rate_diff,False
7,income,nearest_event_rate,3,2.0,"[3654.94, 4671.23)",2,1875.0,500.0,1375.0,0.266667,...,1004.0,375.0,629.0,0.373506,-0.331011,0.106839,0.494298,0.106839,abs_event_rate_diff,False
8,income,nearest_event_rate,4,3.0,"[4671.23, 6271.81)",3,1534.0,338.0,1196.0,0.220339,...,1004.0,375.0,629.0,0.373506,-0.331011,0.153167,0.745964,0.153167,abs_event_rate_diff,False
9,income,nearest_event_rate,5,4.0,"[6271.81, inf)",4,1076.0,152.0,924.0,0.141264,...,1004.0,375.0,629.0,0.373506,-0.331011,0.232242,1.285421,0.232242,abs_event_rate_diff,False


## 13. Transformar a base com múltiplas variáveis

In [16]:
transformed = rb.transform(
    df[features],
    columns=features,
    return_type="dataframe",
)

transformed.head(10)

,bureau_score,income,channel
0,"[560.39, 653.85)","[3654.94, 4671.23)",1
1,"[518.58, 560.39)","[4671.23, 6271.81)",2
2,"[653.85, 736.69)","(-inf, 3242.65)",0
3,"[653.85, 736.69)","(-inf, 3242.65)",2
4,"(-inf, 518.58)","[4671.23, 6271.81)",2
5,"(-inf, 518.58)","[3654.94, 4671.23)",1
6,"[560.39, 653.85)","[6271.81, inf)",0
7,"[560.39, 653.85)","(-inf, 3242.65)",0
8,"[560.39, 653.85)","(-inf, 3242.65)",2
9,"[518.58, 560.39)","[4671.23, 6271.81)",0


## 14. Exportar evidências

O RiskBands pode exportar um bundle com artefatos de auditoria e um relatório HTML narrativo.

Esses arquivos ajudam na revisão por:

- cientistas de dados;
- validação de modelos;
- auditoria;
- governança.

In [17]:
output_dir = Path("riskbands_demo_bundle")
if output_dir.exists():
    import shutil
    shutil.rmtree(output_dir)

rb.export_bundle(output_dir)

print("Bundle exportado para:", output_dir.resolve())
print("Arquivos principais:")
for path in sorted(output_dir.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(output_dir))

Bundle exportado para: D:\0_CienciaDados\1_Frameworks\RiskBands\examples\riskbands_demo_bundle
Arquivos principais:
- audit_report.html
- audit_table.csv
- binnings.json
- feature_tables\bureau_score.csv
- feature_tables\channel.csv
- feature_tables\income.csv
- metadata.json
- missing_decision_log.csv
- missing_merge_candidates.csv
- missing_profile.csv
- missing_transform_fallback_log.csv
- report.csv
- score_details.csv
- score_table.csv
- summary.csv


## 15. Abrir o relatório narrativo

O arquivo `audit_report.html` é pensado para leitura humana e pode ser aberto no navegador.

Ele resume:

- configuração;
- variáveis;
- decisões sobre ausentes;
- candidatos avaliados;
- inventário do bundle;
- limitações.

In [18]:
audit_report_path = output_dir / "audit_report.html"
print("Audit report:", audit_report_path.resolve())
print("Existe?", audit_report_path.exists())

# Em Jupyter local, você pode abrir manualmente o arquivo no navegador.
# Em alguns ambientes, display(HTML(...)) pode funcionar, mas evitamos carregar HTML grande no notebook.

Audit report: D:\0_CienciaDados\1_Frameworks\RiskBands\examples\riskbands_demo_bundle\audit_report.html
Existe? True


## 16. Exemplo opcional com PySpark

Esta célula é opcional. Ela mostra o caminho suportado:

```text
fit pandas -> transform Spark
```

Se PySpark não estiver disponível, a célula apenas informa e segue.

In [19]:
try:
    import pyspark
    from pyspark.sql import SparkSession
    from pyspark.sql import types as T

    spark = (
        SparkSession.builder
        .appName("riskbands-presentation-example")
        .master("local[*]")
        .config("spark.sql.execution.arrow.pyspark.enabled", "false")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .getOrCreate()
    )

    print("Spark:", spark.version)

    # Para categóricas, preferimos schema explícito para que missing chegue como NULL real.
    sample_for_spark = df[features].head(20).copy()
    sample_for_spark["channel"] = sample_for_spark["channel"].astype("object")
    sample_for_spark.loc[sample_for_spark["channel"].isna(), "channel"] = None

    schema = T.StructType([
        T.StructField("bureau_score", T.DoubleType(), True),
        T.StructField("income", T.DoubleType(), True),
        T.StructField("channel", T.StringType(), True),
    ])

    sdf = spark.createDataFrame(sample_for_spark, schema=schema)

    spark_transformed = rb.transform(
        sdf,
        columns=features,
        return_type="dataframe",
    )

    spark_transformed.show(truncate=False)

except Exception as exc:
    print("PySpark opcional não executado neste ambiente.")
    print(type(exc).__name__, exc)

Spark: 3.5.8
PySpark opcional não executado neste ambiente.
Py4JJavaError An error occurred while calling o153.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (DESKTOP-FI3PI2V.bwrouter executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:203)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:331)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.s

## 17. Mensagem final

Este exemplo mostrou um fluxo mínimo:

1. criar ou carregar dados;
2. ajustar binnings;
3. inspecionar cortes e métricas;
4. registrar decisões sobre missing;
5. transformar dados;
6. exportar evidências.

Em um projeto real, o próximo passo seria comparar variáveis, safras, estabilidade temporal e validações out-of-time.